In [ ]:
import sqlite3
import json
import gzip
from collections import Counter, defaultdict
from pathlib import Path

MBTILES_PATH = Path.cwd().parent / 'odseki_map_vector_final_brezZ.mbtiles'
LAYER_NAME = 'odsek'

# Možna imena atributa za GGO v tile feature-jih
GGO_CANDIDATE_FIELDS = ['ggo', 'ggo_id', 'ggo_sifra', 'ggo_code', 'ggo_naziv', 'ggo_name']
ODSEK_FIELD = 'odsek'

# Nastavi npr. 5000 za hitrejši test; None = vsi tile-i
MAX_TILES = None

MBTILES_PATH

PosixPath('/home/tiln/Documents/BarkWatch_Arnes-Hackathon-2026_interface/odseki_map_vector.mbtiles')

In [3]:
# Če manjka knjižnica, odkomentiraj naslednjo vrstico:
%pip install mapbox-vector-tile

import mapbox_vector_tile

print('mapbox_vector_tile OK')

  Using cached shapely-2.1.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.8 kB)
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 3.5 MB/s eta 0:00:00a 0:00:01
Using cached shapely-2.1.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (3.1 MB)
Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
mapbox_vector_tile OK


In [4]:
def maybe_decompress(tile_blob: bytes) -> bytes:
    if tile_blob[:2] == b'\x1f\x8b':
        return gzip.decompress(tile_blob)
    return tile_blob

def first_existing_ggo(props: dict):
    for k in GGO_CANDIDATE_FIELDS:
        if k in props and str(props.get(k, '')).strip() != '':
            return k, str(props.get(k)).strip()
    return None, ''

def analyze_mbtiles(mbtiles_path: Path, layer_name: str, max_tiles=None):
    conn = sqlite3.connect(str(mbtiles_path))
    cur = conn.cursor()

    meta = dict(cur.execute('SELECT name, value FROM metadata').fetchall())
    total_tiles = cur.execute('SELECT COUNT(*) FROM tiles').fetchone()[0]

    q = 'SELECT zoom_level, tile_column, tile_row, tile_data FROM tiles'
    if max_tiles is not None:
        q += f' LIMIT {int(max_tiles)}'

    tile_rows = cur.execute(q).fetchall()

    stats = {
        'tiles_scanned': len(tile_rows),
        'tiles_total': total_tiles,
        'features_total': 0,
        'features_in_layer': 0,
        'missing_odsek': 0,
        'missing_ggo': 0,
        'missing_both': 0,
        'decode_errors': 0,
        'layer_missing_in_tile': 0,
    }

    key_counter = Counter()          # (odsek, ggo_value)
    odsek_counter = Counter()        # odsek only
    ggo_field_counter = Counter()    # which ggo field was used

    key_examples = defaultdict(list)
    odsek_examples = defaultdict(list)

    for z, x, y, blob in tile_rows:
        try:
            raw = maybe_decompress(blob)
            decoded = mapbox_vector_tile.decode(raw)
        except Exception:
            stats['decode_errors'] += 1
            continue

        if layer_name not in decoded:
            stats['layer_missing_in_tile'] += 1
            continue

        feats = decoded[layer_name].get('features', [])
        stats['features_in_layer'] += len(feats)

        for f in feats:
            stats['features_total'] += 1
            props = f.get('properties', {}) or {}

            odsek_val = str(props.get(ODSEK_FIELD, '')).strip()
            ggo_field, ggo_val = first_existing_ggo(props)

            if not odsek_val:
                stats['missing_odsek'] += 1
            if not ggo_val:
                stats['missing_ggo'] += 1
            if (not odsek_val) and (not ggo_val):
                stats['missing_both'] += 1

            if ggo_field:
                ggo_field_counter[ggo_field] += 1

            if odsek_val:
                odsek_counter[odsek_val] += 1
                if len(odsek_examples[odsek_val]) < 3:
                    odsek_examples[odsek_val].append((z, x, y))

            if odsek_val and ggo_val:
                k = (odsek_val, ggo_val)
                key_counter[k] += 1
                if len(key_examples[k]) < 3:
                    key_examples[k].append((z, x, y))

    conn.close()

    duplicate_odsek = {k: v for k, v in odsek_counter.items() if v > 1}
    duplicate_key = {k: v for k, v in key_counter.items() if v > 1}

    return {
        'meta': meta,
        'stats': stats,
        'ggo_field_counter': ggo_field_counter,
        'odsek_counter': odsek_counter,
        'key_counter': key_counter,
        'duplicate_odsek': duplicate_odsek,
        'duplicate_key': duplicate_key,
        'odsek_examples': odsek_examples,
        'key_examples': key_examples,
    }

In [5]:
if not MBTILES_PATH.exists():
    raise FileNotFoundError(f'MBTiles not found: {MBTILES_PATH}')

result = analyze_mbtiles(MBTILES_PATH, LAYER_NAME, max_tiles=MAX_TILES)
stats = result['stats']

print(f'MBTiles: {MBTILES_PATH}')
print(f"Layer: {LAYER_NAME}")
print(f"Tiles scanned: {stats['tiles_scanned']} / {stats['tiles_total']}")
print(f"Features in layer: {stats['features_in_layer']}")
print(f"Missing odsek: {stats['missing_odsek']}")
print(f"Missing ggo(any candidate): {stats['missing_ggo']}")
print(f"Missing both: {stats['missing_both']}")
print(f"Decode errors: {stats['decode_errors']}")

print('\nGGO field usage:')
for k, v in result['ggo_field_counter'].most_common():
    print(f'- {k}: {v}')

print('\nUniqueness check:')
print(f"- Duplicate odsek values: {len(result['duplicate_odsek'])}")
print(f"- Duplicate (odsek, ggo) keys: {len(result['duplicate_key'])}")

MBTiles: /home/tiln/Documents/BarkWatch_Arnes-Hackathon-2026_interface/odseki_map_vector.mbtiles
Layer: odsek
Tiles scanned: 10086 / 10086
Features in layer: 446071
Missing odsek: 0
Missing ggo(any candidate): 446071
Missing both: 0
Decode errors: 0

GGO field usage:

Uniqueness check:
- Duplicate odsek values: 35688
- Duplicate (odsek, ggo) keys: 0


In [ ]:
print('# Top duplicate odsek (same odsek appears multiple times)')
for odsek, n in sorted(result['duplicate_odsek'].items(), key=lambda kv: (-kv[1], kv[0]))[:20]:
    ex = result['odsek_examples'][odsek]
    print(f'- {odsek}: {n}x | sample tiles: {ex}')

print('\n# Top duplicate (odsek, ggo) keys')
for key, n in sorted(result['duplicate_key'].items(), key=lambda kv: (-kv[1], kv[0]))[:20]:
    ex = result['key_examples'][key]
    print(f'- {key}: {n}x | sample tiles: {ex}')

In [ ]:
print('# Hitri zaključek')
if stats['missing_ggo'] > 0:
    print('- V MBTiles ni konsistentnega GGO atributa za vse feature-je.')
else:
    print('- Vsi feature-ji imajo GGO (vsaj en kandidatni atribut).')

if len(result['duplicate_key']) > 0:
    print('- Ključ <odsek, ggo> NI enoličen v MBTiles.')
else:
    print('- Ključ <odsek, ggo> je enoličen (po skeniranih tile-ih).')

if len(result['duplicate_odsek']) > 0:
    print('- `odsek` sam NI enoličen, zato lahko highlight/locate skoči na napačen poligon.')
else:
    print('- `odsek` je enoličen v MBTiles.')